In [1]:
import kagglehub
dataset_path = kagglehub.dataset_download("salader/dogs-vs-cats")

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Conv1D,Conv2D,Flatten,Dense,MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D, GlobalMaxPooling2D, BatchNormalization, Dropout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, r2_score
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.datasets.mnist import load_data
from keras.datasets.fashion_mnist import load_data
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPool2D, Flatten, Dense, Dropout
from tensorflow.keras.models import Sequential
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [3]:
!ln -s $dataset_path"/train" "/content/"
!ln -s $dataset_path"/test" "/content/"

In [4]:
train_ds=keras.utils.image_dataset_from_directory(
    directory="/content/train",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256,256)
)
validation_ds=keras.utils.image_dataset_from_directory(
    directory="/content/test",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256,256)
)
def normalize_process(image,label):
  image=tf.cast(image/255.,tf.float32)
  return image, label
train_ds=train_ds.map(normalize_process)
validation_ds=validation_ds.map(normalize_process)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


# AlexNet

In [5]:
amodel = Sequential()
amodel.add(Conv2D(96, kernel_size=(11, 11), strides=4, padding="valid", activation="relu", input_shape=(256, 256, 3)))
amodel.add(BatchNormalization())
amodel.add(MaxPooling2D(pool_size=(3, 3), strides=2))
amodel.add(Conv2D(256, kernel_size=(5, 5), padding="same", activation="relu"))
amodel.add(BatchNormalization())
amodel.add(MaxPooling2D(pool_size=(3, 3), strides=2))
amodel.add(Conv2D(384, kernel_size=(3, 3), padding="same", activation="relu"))
amodel.add(BatchNormalization())
amodel.add(Conv2D(384, kernel_size=(3, 3), padding="same", activation="relu"))
amodel.add(BatchNormalization())
amodel.add(Conv2D(256, kernel_size=(3, 3), padding="same", activation="relu"))
amodel.add(BatchNormalization())
amodel.add(MaxPooling2D(pool_size=(3, 3), strides=2))
amodel.add(Flatten())
amodel.add(Dense(4096, activation="relu"))
amodel.add(Dropout(0.5))
amodel.add(Dense(4096, activation="relu"))
amodel.add(Dropout(0.5))
amodel.add(Dense(1, activation="sigmoid"))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
amodel.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 62, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 30, 30, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 30, 30, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30, 30, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 14, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 14, 14, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 14, 384)    │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 14, 14, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 14, 14, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         4,097 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,290,945 (222.36 MB)

 Trainable params: 58,288,193 (222.35 MB)

 Non-trainable params: 2,752 (10.75 KB)

In [7]:
amodel.compile(optimizer="adam", metrics=["accuracy"], loss="binary_crossentropy")

In [ ]:
ahistory=amodel.fit(train_ds, epochs=10, validation_data=validation_ds)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4777s 8s/step - accuracy: 0.5340 - loss: 3.7420 - val_accuracy: 0.5314 - val_loss: 0.8680
Epoch 2/10
 81/625 ━━━━━━━━━━━━━━━━━━━━ 1:04:05 7s/step - accuracy: 0.5865 - loss: 0.6988

In [ ]:
plt.plot(ahistory.history["loss"], label="loss")
plt.plot(ahistory.history["val_loss"], label="val_loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(ahistory.history["accuracy"], label="accuracy")
plt.plot(ahistory.history["val_accuracy"], label="val_accuracy")
plt.legend()
plt.show()